# 🛒 LangGraph Procurement Review Flow

## What We're Building

A procurement review workflow that:
1. **Collects vendor proposals** — gathers structured bids from multiple vendors
2. **Scores each vendor** — evaluates on price, quality, delivery, and terms
3. **Compares options side-by-side** — builds a comparison matrix
4. **Summarizes with LLM** — produces a final recommendation memo

## Architecture

```
Vendor Proposals
      │
      ▼
┌────────────┐    ┌────────────┐    ┌────────────┐    ┌──────────────┐
│  Collect    │ ──▶│  Score     │ ──▶│  Compare   │ ──▶│  Summarize   │
│  Proposals  │    │  Vendors   │    │  Options   │    │  & Recommend │
└────────────┘    └────────────┘    └────────────┘    └──────────────┘
```

## Stack
- **LangGraph** — orchestrates the multi-step review flow
- **Ollama** — local LLM for vendor comparison and summary generation
- **pandas** — tabular comparison of vendor proposals

## Step 1 — Imports & Configuration

In [ ]:
import json
import pandas as pd
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:4b", temperature=0.2)
print("All imports successful!")

## Step 2 — Sample Vendor Proposals

We simulate three vendor bids for a bulk office-supply procurement.
Each proposal includes price, estimated delivery days, warranty terms,
a quality rating (1-10), and any special notes.

In [ ]:
VENDOR_PROPOSALS = [
    {
        "vendor": "AlphaSupply Co.",
        "unit_price_usd": 12.50,
        "quantity": 5000,
        "delivery_days": 7,
        "warranty_months": 12,
        "quality_rating": 8,
        "payment_terms": "Net-30",
        "notes": "ISO 9001 certified. Free shipping on orders over $50K."
    },
    {
        "vendor": "BetaGoods Inc.",
        "unit_price_usd": 10.75,
        "quantity": 5000,
        "delivery_days": 14,
        "warranty_months": 6,
        "quality_rating": 6,
        "payment_terms": "Net-15",
        "notes": "Budget option. Bulk discount of 5% on reorders."
    },
    {
        "vendor": "GammaPremium Ltd.",
        "unit_price_usd": 15.00,
        "quantity": 5000,
        "delivery_days": 5,
        "warranty_months": 24,
        "quality_rating": 9,
        "payment_terms": "Net-45",
        "notes": "Premium quality. Dedicated account manager. 24/7 support."
    }
]

print(f"Loaded {len(VENDOR_PROPOSALS)} vendor proposals")
for v in VENDOR_PROPOSALS:
    print(f"  - {v['vendor']}: ${v['unit_price_usd']}/unit, {v['delivery_days']}d delivery")

## Step 3 — Define State & Scoring Weights

The state carries vendor data through the graph.
Scoring weights let the procurement team emphasize what matters most.

In [ ]:
class ProcurementState(TypedDict):
    proposals_json: str
    scored_table: str
    comparison_analysis: str
    final_recommendation: str

WEIGHTS = {
    "price": 0.30,
    "quality": 0.25,
    "delivery": 0.20,
    "warranty": 0.15,
    "terms": 0.10,
}

print("State and weights defined")
print("Weights:", WEIGHTS)

## Step 4 — Define Graph Nodes

| Node | Role |
|------|------|
| `collect_proposals` | Parse and validate vendor data |
| `score_vendors` | Compute weighted scores for each vendor |
| `compare_options` | LLM generates a side-by-side comparison |
| `summarize_recommendation` | LLM produces a final recommendation memo |

In [ ]:
def collect_proposals(state: ProcurementState) -> dict:
    """Parse and validate vendor proposals."""
    print("📥 Node: collect_proposals — Validating vendor bids...")
    proposals = json.loads(state["proposals_json"])
    required_keys = {"vendor", "unit_price_usd", "quantity", "delivery_days",
                     "warranty_months", "quality_rating", "payment_terms"}
    for p in proposals:
        missing = required_keys - set(p.keys())
        if missing:
            print(f"  ⚠️ {p.get('vendor', 'UNKNOWN')}: missing {missing}")
        else:
            total = p["unit_price_usd"] * p["quantity"]
            print(f"  ✓ {p['vendor']}: ${total:,.0f} total")
    return {}


def score_vendors(state: ProcurementState) -> dict:
    """Score each vendor on a 0-100 weighted scale."""
    print("📊 Node: score_vendors — Computing weighted scores...")
    proposals = json.loads(state["proposals_json"])
    prices = [p["unit_price_usd"] for p in proposals]
    qualities = [p["quality_rating"] for p in proposals]
    deliveries = [p["delivery_days"] for p in proposals]
    warranties = [p["warranty_months"] for p in proposals]
    terms_days = [int(p["payment_terms"].split("-")[1]) for p in proposals]

    def norm(val, vals, invert=False):
        r = max(vals) - min(vals)
        if r == 0:
            return 1.0
        n = (val - min(vals)) / r
        return 1 - n if invert else n

    rows = []
    for i, p in enumerate(proposals):
        weighted = (
            WEIGHTS["price"] * norm(prices[i], prices, invert=True) +
            WEIGHTS["quality"] * norm(qualities[i], qualities) +
            WEIGHTS["delivery"] * norm(deliveries[i], deliveries, invert=True) +
            WEIGHTS["warranty"] * norm(warranties[i], warranties) +
            WEIGHTS["terms"] * norm(terms_days[i], terms_days)
        ) * 100
        rows.append({
            "Vendor": p["vendor"],
            "Unit Price": f"${p['unit_price_usd']}",
            "Total Cost": f"${p['unit_price_usd'] * p['quantity']:,.0f}",
            "Delivery": f"{p['delivery_days']}d",
            "Quality": f"{p['quality_rating']}/10",
            "Warranty": f"{p['warranty_months']}mo",
            "Terms": p["payment_terms"],
            "Score": round(weighted, 1)
        })
    df = pd.DataFrame(rows).sort_values("Score", ascending=False)
    table_str = df.to_string(index=False)
    print(f"     Scored {len(proposals)} vendors")
    return {"scored_table": table_str}


compare_prompt = ChatPromptTemplate.from_template(
    """You are a procurement analyst. Compare these vendor proposals side-by-side.

Scored Comparison:
{scored_table}

Raw Proposal Details:
{proposals}

Provide:
1. STRENGTHS of each vendor (2-3 bullet points each)
2. RISKS or WEAKNESSES for each vendor
3. Which vendor is best for: (a) cost savings, (b) quality-first, (c) fastest delivery

Keep the analysis concise and data-driven."""
)


def compare_options(state: ProcurementState) -> dict:
    """Use LLM to generate a side-by-side vendor comparison."""
    print("🔍 Node: compare_options — Generating LLM comparison...")
    chain = compare_prompt | llm | StrOutputParser()
    analysis = chain.invoke({
        "scored_table": state["scored_table"],
        "proposals": state["proposals_json"]
    })
    return {"comparison_analysis": analysis}


recommend_prompt = ChatPromptTemplate.from_template(
    """You are a senior procurement manager. Write a recommendation memo.

Scored Table:
{scored_table}

Analyst Comparison:
{comparison}

Write a procurement recommendation memo with:
- RECOMMENDED VENDOR and why
- KEY FACTORS that drove the decision
- RISKS to monitor
- SUGGESTED NEXT STEPS

Keep it professional and actionable."""
)


def summarize_recommendation(state: ProcurementState) -> dict:
    """Produce the final recommendation memo."""
    print("📝 Node: summarize_recommendation — Writing final memo...")
    chain = recommend_prompt | llm | StrOutputParser()
    memo = chain.invoke({
        "scored_table": state["scored_table"],
        "comparison": state["comparison_analysis"]
    })
    return {"final_recommendation": memo}


print("All nodes defined")

## Step 5 — Build & Compile the Graph

In [ ]:
workflow = StateGraph(ProcurementState)

workflow.add_node("collect_proposals", collect_proposals)
workflow.add_node("score_vendors", score_vendors)
workflow.add_node("compare_options", compare_options)
workflow.add_node("summarize_recommendation", summarize_recommendation)

workflow.set_entry_point("collect_proposals")
workflow.add_edge("collect_proposals", "score_vendors")
workflow.add_edge("score_vendors", "compare_options")
workflow.add_edge("compare_options", "summarize_recommendation")
workflow.add_edge("summarize_recommendation", END)

app = workflow.compile()
print("Procurement review workflow compiled")

## Step 6 — Run the Full Procurement Review

In [ ]:
result = app.invoke({
    "proposals_json": json.dumps(VENDOR_PROPOSALS),
    "scored_table": "",
    "comparison_analysis": "",
    "final_recommendation": "",
})

print("\n" + "="*60)
print("📊 SCORED VENDOR TABLE")
print("="*60)
print(result["scored_table"])

print("\n" + "="*60)
print("🔍 COMPARISON ANALYSIS")
print("="*60)
print(result["comparison_analysis"])

print("\n" + "="*60)
print("📝 FINAL RECOMMENDATION MEMO")
print("="*60)
print(result["final_recommendation"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Structured data flow** | Vendor proposals flow as JSON through typed state |
| **Deterministic scoring** | Weighted normalization gives reproducible scores |
| **LLM comparison** | Ollama analyzes trade-offs humans might miss |
| **LLM summarization** | Final memo is actionable and professional |

## Why LangGraph for Procurement?

- Each review stage is a clear, testable node
- Easy to add nodes: compliance checks, budget validation, approval gates
- State carries structured data — no lost context between steps
- LLM augments human judgment rather than replacing it

## 🔧 Extensions

- **HITL approval gate**: Add `interrupt_before` so a manager reviews before finalizing
- **Historical comparison**: Pull past vendor performance from a database
- **Multi-category RFP**: Handle proposals across different product categories
- **PDF export**: Generate a formatted procurement report